In [43]:
import os
os.environ["PYTHONNET_RUNTIME"] = "coreclr"
os.environ["DOTNET_SYSTEM_DRAWING_USE_GDIPLUS"] = "1"

import clr
import numpy as np
import time
from datetime import timedelta
from pythonnet import load
from System import Environment, Array, String

# Load .NET Core runtime
load("coreclr")

# Try to import System.IO; fallback to os.chdir if it fails
try:
    from System.IO import Directory, Path, File
    use_dotnet_dir = True
except Exception as e:
    print(f"Note: System.IO import failed ({e}), using os.chdir instead")
    import os
    use_dotnet_dir = False

# Define DWSIM path
dwsimpath = "/usr/local/lib/dwsim/"

# Set working directory
if use_dotnet_dir:
    Directory.SetCurrentDirectory(dwsimpath)
else:
    os.chdir(dwsimpath)

# Add DWSIM assemblies
clr.AddReference(dwsimpath + "CapeOpen.dll")
clr.AddReference(dwsimpath + "DWSIM.Automation.dll")
clr.AddReference(dwsimpath + "DWSIM.Interfaces.dll")
clr.AddReference(dwsimpath + "DWSIM.GlobalSettings.dll")
clr.AddReference(dwsimpath + "DWSIM.SharedClasses.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.dll")
clr.AddReference(dwsimpath + "DWSIM.UnitOperations.dll")
clr.AddReference(dwsimpath + "DWSIM.Inspector.dll")
clr.AddReference(dwsimpath + "System.Buffers.dll")
clr.AddReference(dwsimpath + "DWSIM.Thermodynamics.ThermoC.dll")

# Now import DWSIM types
from DWSIM.Interfaces.Enums.GraphicObjects import ObjectType
from DWSIM.Thermodynamics import Streams, PropertyPackages
from DWSIM.UnitOperations import UnitOperations
from DWSIM.Automation import Automation3
from DWSIM.GlobalSettings import Settings

print("DWSIM imports successful!")

DWSIM imports successful!


In [44]:
# Create an instance of the Automation3 class from the DWSIM.Automation module
# This class provides methods for automating tasks in DWSIM, such as creating and manipulating flowsheets
interf = Automation3()

In [45]:
# Creates a flowsheet
sim = interf.CreateFlowsheet()

In [46]:
# Add Compounds
sim.AddCompound("Air")
sim.AddCompound("Salt")

In [47]:
one = sim.AddFlowsheetObject('Material Stream','Stream-1')
two = sim.AddFlowsheetObject('Material Stream','Stream-2')
Ball_Mill = sim.AddFlowsheetObject('Python Script','Ball Mill')

In [48]:
# property package
Thermo_Package = sim.CreateAndAddPropertyPackage("Raoult's Law")

In [49]:
one = one.GetAsObject()
two = two.GetAsObject()
Ball_Mill = Ball_Mill.GetAsObject()

In [50]:
sim.ConnectObjects(one.GraphicObject, Ball_Mill.GraphicObject, -1, -1)
sim.ConnectObjects(Ball_Mill.GraphicObject, two.GraphicObject, -1, -1)

In [51]:
sim.AutoLayout()

In [52]:
one.SetOverallComposition(Array[float]([0.2, 0.8]))
one.SetTemperature(298.15)
one.SetPressure(101325)
one.SetMassFlow(1.48657)

'Stream-1: mass flow set to 1.48657 kg/s'

In [53]:
Ball_Mill.ScriptText = open("/workspace/00 FlowSheet Automation/19 Custom_UO_Python/ball_mill_script.py").read()

In [54]:
# InputVariables / InputStringVariables are plain .NET Dictionary<string, object>
Ball_Mill.InputVariables.Add("ResidenceTime", 600.0)
Ball_Mill.InputVariables.Add("WorkIndex", 12.5)
Ball_Mill.InputVariables.Add("Alpha", 1.0)
Ball_Mill.InputVariables.Add("aT", 0.05)
Ball_Mill.InputVariables.Add("Mu", 4.0)
Ball_Mill.InputVariables.Add("Gamma", 0.8)
Ball_Mill.InputVariables.Add("Beta", 3.5)
Ball_Mill.InputVariables.Add("Phi", 0.72)
Ball_Mill.InputVariables.Add("MillDiameter", 1.5)
Ball_Mill.InputVariables.Add("MillLength", 3.0)
Ball_Mill.InputVariables.Add("BallFilling", 0.35)
Ball_Mill.InputVariables.Add("SolidIndex", 0.0)

Ball_Mill.InputStringVariables.Add("SizeRange", "[10, 25, 45, 75, 150, 300, 600, 1200]")
Ball_Mill.InputStringVariables.Add("FeedMassFrac", "[0.0, 0.0, 0.05, 0.10, 0.30, 0.30, 0.20, 0.05]")
Ball_Mill.InputStringVariables.Add("FlowModel", '"CSTR"')

In [55]:
Ball_Mill.EmbeddedImageData = '/workspace/00 FlowSheet Automation/19 Custom_UO_Python/ball mill icon.png'

In [56]:
# request a calculation
errors = interf.CalculateFlowsheet4(sim)
list(errors)

[]

In [57]:
# save file

fileNameToSave = Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.Desktop), "/workspace/00 FlowSheet Automation/19 Custom_UO_Python/19 Custom_UO_Python_Automation.dwxmz")

interf.SaveFlowsheet(sim, fileNameToSave, True)